# Week 3 lab: find what the file is hiding

A file lands on your desk. Someone will make a decision with it at the next
shift handover. Your job today is not to clean it. Your job is to find out what
it will not support, and to leave evidence another person can check.

Three rules hold for the whole lab:

1. **Never edit the raw file.** A fault you delete is a fault nobody else can see.
2. **Report, do not repair.** A correction needs a source, an approver, and a record.
3. **Name your evidence.** "Row 44" beats "some temperatures look high".

Your file is generated from your username, so the faults in it are yours. The
person next to you has different ones. Comparing answers is useful; copying them
will not work.

## Setup

Run this in the terminal first, using your own GitHub Classroom username:

```bash
python src/make_variant.py --student-id your-github-username
```

Then set `STUDENT_ID` below to the same value.

In [ ]:
from pathlib import Path
import hashlib
import sys

import pandas as pd
from IPython.display import display

# Put your GitHub Classroom username here. Your file is generated from it.
STUDENT_ID = "REPLACE_WITH_YOUR_USERNAME"


def locate_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "validate_data.py").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from the week03 project folder.")


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(65536), b""):
            digest.update(block)
    return digest.hexdigest()


PROJECT_ROOT = locate_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from make_variant import slugify  # noqa: E402

DATA_PATH = PROJECT_ROOT / "data" / "raw" / f"shift_log_{slugify(STUDENT_ID)}.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH.name} is missing. Run this in the terminal first:\n"
        f"    python src/make_variant.py --student-id {STUDENT_ID}"
    )

df = pd.read_csv(DATA_PATH)
print(f"File:    {DATA_PATH.name}")
print(f"Rows:    {len(df)}   Columns: {len(df.columns)}")
print(f"SHA-256: {file_sha256(DATA_PATH)}")
print("\nCopy the file name and fingerprint into docs/data_quality_report.md now.")

## Step 1. Write the contract before you look

Before any code, answer these in the cell below. You are writing the standard
that the rest of the lab measures the file against. If you skip this, every
check you write later is just an opinion.

- What does one row represent?
- Which columns together identify one row uniquely?
- How many records should each line have, and over what period?
- Which columns can never be negative, and why is that true from their meaning
  rather than from the numbers you happen to see?

**Your contract:**

- One row represents:
- The business key is:
- Expected coverage:
- Columns that cannot be negative, and why:

In [ ]:
display(df.head(6))

schema = pd.DataFrame({
    "column": df.columns,
    "dtype_on_load": [str(value) for value in df.dtypes],
    "missing": [int(df[column].isna().sum()) for column in df.columns],
    "distinct": [int(df[column].nunique(dropna=True)) for column in df.columns],
})
display(schema)

## Step 2. Hunt, before the validator helps you

Work through the four cells below **without running the validator**. You are
looking for records you would not want a supervisor to act on.

Keep a list as you go. You will be asked to commit to it in a moment.

### Completeness

In [ ]:
# TODO Which columns have missing values, and in which records?
# Print a count per column, then show the affected rows themselves.

### Uniqueness

In [ ]:
# TODO Does any record repeat another exactly?
# TODO Does any line-hour appear more than once with DIFFERENT values?
# These are two different questions. Answer both, and show the rows.

### Validity

Separate two ideas that feel similar and are not:

- **impossible** under the contract, such as rejecting more units than you made;
- **surprising**, such as a high motor temperature, which may be an error, a
  real event, or a line running differently.

Only the first kind can be judged from the file alone.

In [ ]:
# TODO Start with df.describe() and read the min and max of every column.
# Then separate what you find into two groups and print each one:
#   impossible under the data contract  (a value that cannot be true)
#   surprising but possible             (a value that needs a human to judge)

### Coverage and time

In [ ]:
# TODO Does every timestamp parse? Show any that do not.
# TODO Does each line have 24 distinct hours? Count unique line-hours, not rows.
# TODO Does any shift label disagree with its hour? Day is 06:00-17:59.

## Step 3. Commit to your list

**Stop here and fill in section 3 of `docs/data_quality_report.md`** with the
records you found, before running the next cell.

This is the point of the exercise. Writing your list down first is what makes
the next cell informative rather than reassuring.

## Step 4. Compare against the checks

The validator runs the rules that are written so far. Unwritten ones are
reported as TODO rather than stopping the run.

In [ ]:
from validate_data import validate

payload = validate(DATA_PATH, skip_unwritten=True)
summary = pd.DataFrame(payload["checks"])[["name", "passed", "observed", "affected_rows"]]
display(summary)
print("Not written yet:", payload["unwritten_checks"] or "none")
print("Overall status:", payload["status"])

**Answer before moving on:**

- Which records did the code find that you missed?
- Which did you find that no check covers yet? That is a check you should write.
- Did any single fault trip more than one check? Is that one problem or two?

**Your answer:**

## Step 5. Write the missing checks

Move to the terminal and work in `src/validate_data.py`:

```bash
pytest -q
```

Six checks are marked TODO and six tests are failing. Write one check, re-run,
and repeat until the tests pass. Then come back and run the cell below.

In [ ]:
payload = validate(DATA_PATH)
summary = pd.DataFrame(payload["checks"])[["name", "passed", "observed", "affected_rows"]]
display(summary)
print("Failed checks:", payload["failed_checks"])
print("Overall status:", payload["status"])

assert not payload["unwritten_checks"], "Some checks are still TODO. Run pytest -q."
print("\nAll twelve checks are written. Now run the script from the terminal so it")
print("writes artifacts/validation_results.json, and commit that file.")

## Step 6. The row that checks cannot settle

One record in your file is unusual rather than impossible: the high motor
temperature. A check can flag it. A check cannot tell you what it means.

**Answer below:**

- Name three explanations for that reading, at least one of which is not a fault.
- What single piece of evidence would separate them?
- Who holds that evidence?
- Until you have it, what may the handover discussion still use this file for?

**Your answer:**

## Next

1. Finish `docs/data_quality_report.md` with what you now know.
2. Do the attack round with your partner and fill in `docs/attack_log.md`.
3. Make the approval decision in `docs/dataset_approval.md`.
4. Commit and push.